# Tutorial 5: Text Embeddings

This notebook demonstrates how to generate **text embeddings** with `embpy`.
Text embeddings are useful for encoding gene descriptions, paper abstracts,
pathway annotations, or any free-text biological metadata into a shared
embedding space.

Available text embedding models:

| Model key | Architecture | Dim |
|---|---|---|
| `minilm_l6_v2` | MiniLM-L6-v2 (sentence-transformers) | 384 |
| `bert_base_uncased` | BERT base | 768 |

These are general-purpose text encoders. You can embed gene descriptions,
pathway names, phenotype annotations, or any text you want to place in
the same vector space as other perturbation embeddings.

In [ ]:
import logging
logging.basicConfig(level=logging.WARNING)

In [ ]:
from embpy.embedder import BioEmbedder

embedder = BioEmbedder(device="auto")
print(f"Device: {embedder.device}")

## 1. Embed a single text string

In [ ]:
MODEL = "minilm_l6_v2"  # lightweight and fast

text = "TP53 encodes tumor protein p53 which regulates cell cycle and apoptosis."

emb = embedder.embed_text(text, model=MODEL, pooling_strategy="mean")

print(f"Shape: {emb.shape}")
print(f"Dtype: {emb.dtype}")
print(f"First 10 dims: {emb[:10]}")

## 2. Embed gene descriptions via GeneResolver

Combine the `GeneResolver` (which fetches gene descriptions from MyGene.info)
with the text embedder for a fully automated gene → text → embedding pipeline.

In [ ]:
from embpy.resources.gene_resolver import GeneResolver

resolver = GeneResolver(auto_download=False)

genes = ["TP53", "BRCA1", "EGFR", "MYC"]
descriptions = {}

for g in genes:
    desc = resolver.get_gene_description(g, id_type="symbol")
    if desc:
        descriptions[g] = desc
        print(f"  {g}: {desc[:80]}…")
    else:
        print(f"  {g}: (no description found)")

In [ ]:
import numpy as np

# Embed all descriptions
desc_embeddings = {}
for gene, desc in descriptions.items():
    desc_embeddings[gene] = embedder.embed_text(desc, model=MODEL, pooling_strategy="mean")

for gene, emb in desc_embeddings.items():
    print(f"  {gene:8s} → dim {emb.shape[0]}")

## 3. Batch text embedding

In [ ]:
texts = [
    "Tumor suppressor gene involved in DNA damage response.",
    "Oncogene driving cell proliferation via the RAS-MAPK pathway.",
    "Kinase inhibitor blocking EGFR signaling.",
    "Anti-inflammatory drug that inhibits cyclooxygenase.",
]

batch_embs = embedder.embed_texts_batch(
    texts=texts,
    model=MODEL,
    pooling_strategy="mean",
)

for t, emb in zip(texts, batch_embs):
    if emb is not None:
        print(f"  \"{t[:50]}…\" → dim {emb.shape[0]}")
    else:
        print(f"  \"{t[:50]}…\" → FAILED")

## 4. Semantic similarity with text embeddings

Text embeddings capture semantic meaning, so related descriptions should
have high cosine similarity.

In [ ]:
from numpy.linalg import norm

def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

emb_a = embedder.embed_text(
    "TP53 is a tumor suppressor that induces apoptosis.",
    model=MODEL, pooling_strategy="mean"
)
emb_b = embedder.embed_text(
    "The p53 protein regulates the cell cycle and prevents cancer.",
    model=MODEL, pooling_strategy="mean"
)
emb_c = embedder.embed_text(
    "Aspirin is a non-steroidal anti-inflammatory drug.",
    model=MODEL, pooling_strategy="mean"
)

print(f"TP53 description vs p53 paraphrase:  {cosine_sim(emb_a, emb_b):.4f}")
print(f"TP53 description vs aspirin:          {cosine_sim(emb_a, emb_c):.4f}")
print(f"p53 paraphrase  vs aspirin:           {cosine_sim(emb_b, emb_c):.4f}")

## 5. Comparing text models

In [ ]:
sample = "BRCA1 is involved in DNA double-strand break repair."

for m in ["minilm_l6_v2", "bert_base_uncased"]:
    try:
        e = embedder.embed_text(sample, model=m, pooling_strategy="mean")
        print(f"  {m:20s} → dim {e.shape[0]}, L2 = {np.linalg.norm(e):.2f}")
    except Exception as exc:
        print(f"  {m:20s} → {exc}")

## 6. Using gene descriptions for text-based gene embeddings

You can also generate gene embeddings directly through `embed_gene`
when using a text model. The embedder will automatically fetch a textual
description of the gene and embed that.

In [ ]:
emb_gene_text = embedder.embed_gene(
    identifier="TP53",
    model="minilm_l6_v2",
    id_type="symbol",
    organism="human",
    pooling_strategy="mean",
)
print(f"Gene text embedding shape: {emb_gene_text.shape}")

## Summary

| What | How |
|---|---|
| Single text embedding | `embed_text("some text", model)` |
| Batch text embedding | `embed_texts_batch(texts, model)` |
| Gene description embedding | `embed_gene("TP53", model="minilm_l6_v2")` |
| Fetch gene description | `GeneResolver.get_gene_description()` |

Next: [Tutorial 6 – Combined Analysis](06_combined_analysis.ipynb)